# Eval Notebook — Pivot: ปังปัง (adapted from Session 3)

ทำตามข้อ 3.2:
1. เตรียม ground truth: 10 คำถาม + chunk ที่ "ถูก" สำหรับแต่ละคำถาม (manual labeling)
2. รับ retrieval จาก app.py สำหรับแต่ละคำถาม top-k=3
3. คำนวณ precision@3
4. คำนวณ recall@3
5. plot histogram ของ similarity score ของ top-1


In [ ]:
# ติดตั้ง/import ที่จำเป็น (ถ้ายังไม่มีให้ปลดคอมเมนต์บรรทัดล่าง)
# !pip install sentence-transformers faiss-cpu matplotlib

import sys
sys.path.append("..")  # ให้ import app.py ได้ถ้า eval.ipynb อยู่ในโฟลเดอร์ย่อย

from app import build_index, retrieve_top_k
import matplotlib.pyplot as plt


In [ ]:
model, index, chunks = build_index()
print(f"โหลด knowledge base แล้ว: {len(chunks)} chunks")
for i, c in enumerate(chunks):
    print(i, "-", c[:60].replace("\n", " "))


## Step 1: Ground truth

ดู index ของ chunk จาก cell ด้านบน แล้วระบุว่าคำถามแต่ละข้อ "ควร" retrieve chunk ไหน (manual labeling)
แก้ `ground_truth` ด้านล่างให้ตรงกับ knowledge base จริงของกลุ่มคุณ (ตัวอย่างนี้ใช้ menu_kb.md ตัวอย่าง)

In [ ]:
# แก้ไขให้ตรงกับคำถาม + chunk index ของ knowledge base ใหม่ (ปังปัง)
# หมายเหตุ: menu_kb.md ปรับเป็นเมนูขนมปังปิ้ง แต่โครงสร้าง chunk ยังเหมือน MilkLab เดิม
# (header เมนู + รายการเมนู 5 อย่าง + ที่ตั้งร้าน + FAQ) เพื่อให้ pattern เทียบกันได้ง่าย
# ลำดับ chunk ปัจจุบัน: 0=หัวข้อเมนูขนมปังปิ้ง, 1=ปังปิ้งเนยนมสด, 2=ปังปิ้งไข่หวาน,
# 3=ปังปิ้งช็อกโกแลต, 4=ปังปิ้งสังขยาใบเตย, 5=ปังปิ้งพีนัทบัตเตอร์กล้วย, 6=ที่ตั้งร้าน, 7=FAQ ลูกค้า
ground_truth = [
    {"question": "ปังปิ้งเนยนมสดราคาเท่าไหร่", "correct_chunks": [1]},
    {"question": "ปังปิ้งไข่หวานใส่อะไรบ้าง", "correct_chunks": [2]},
    {"question": "ปังปิ้งช็อกโกแลตแพ้นมไหม", "correct_chunks": [3]},
    {"question": "ร้านเปิดกี่โมงถึงกี่โมง", "correct_chunks": [6]},
    {"question": "มีเมนูไม่ใส่ถั่วแนะนำไหม", "correct_chunks": [7]},
    {"question": "จ่ายเงินผ่าน PromptPay ได้ไหม", "correct_chunks": [7]},
    {"question": "สั่งล่วงหน้าได้ไหม", "correct_chunks": [7]},
    {"question": "เมนูไหนถูกที่สุด", "correct_chunks": [1, 2, 3, 4, 5]},
    {"question": "ร้านตั้งอยู่ที่ไหน", "correct_chunks": [6]},
    {"question": "ปังปิ้งพีนัทบัตเตอร์กล้วยราคาเท่าไหร่", "correct_chunks": [5]},
]
len(ground_truth)


## Step 2: รับ retrieval สำหรับแต่ละคำถาม (top-k=3)

In [ ]:
K = 3
eval_results = []

for item in ground_truth:
    retrieved = retrieve_top_k(item["question"], model, index, chunks, k=K)
    retrieved_indices = [chunks.index(c) for c, _ in retrieved]
    top1_score = retrieved[0][1] if retrieved else 0.0
    eval_results.append({
        "question": item["question"],
        "correct_chunks": item["correct_chunks"],
        "retrieved_indices": retrieved_indices,
        "top1_score": top1_score,
    })

for r in eval_results:
    print(r["question"], "-> retrieved:", r["retrieved_indices"], "| correct:", r["correct_chunks"], "| top1 score:", round(r["top1_score"], 3))


## Step 3+4: precision@3 และ recall@3

- precision@3 = (จำนวน chunk ที่ retrieve ถูก / 3) เฉลี่ยจาก 10 คำถาม
- recall@3 = (จำนวน ground-truth chunk ที่อยู่ใน top-3 / จำนวน ground-truth chunk) เฉลี่ยจาก 10 คำถาม

In [ ]:
def precision_at_k(retrieved_indices, correct_chunks):
    hits = len(set(retrieved_indices) & set(correct_chunks))
    return hits / len(retrieved_indices) if retrieved_indices else 0.0

def recall_at_k(retrieved_indices, correct_chunks):
    hits = len(set(retrieved_indices) & set(correct_chunks))
    return hits / len(correct_chunks) if correct_chunks else 0.0

precisions = [precision_at_k(r["retrieved_indices"], r["correct_chunks"]) for r in eval_results]
recalls = [recall_at_k(r["retrieved_indices"], r["correct_chunks"]) for r in eval_results]

mean_precision = sum(precisions) / len(precisions)
mean_recall = sum(recalls) / len(recalls)

print(f"precision@{K} เฉลี่ย: {mean_precision:.3f}")
print(f"recall@{K} เฉลี่ย: {mean_recall:.3f}")


## Step 5: plot histogram ของ similarity score (top-1)

In [ ]:
top1_scores = [r["top1_score"] for r in eval_results]

plt.figure(figsize=(6, 4))
plt.hist(top1_scores, bins=10, range=(0, 1), edgecolor="black")
plt.xlabel("Top-1 similarity score")
plt.ylabel("จำนวนคำถาม")
plt.title("Histogram: Top-1 Similarity Score ต่อคำถาม")
plt.tight_layout()
plt.savefig("top1_score_histogram.png", dpi=150)
plt.show()


## Reflection

_(เขียนสะท้อนผลตรงนี้ก่อนส่ง — เช่น precision/recall สูงหรือต่ำเพราะอะไร, chunk ไหน retrieve ผิดบ่อย, จะปรับ chunking หรือ embedding อย่างไรให้ดีขึ้น)_